# Assignment 5: Word2Vec, FastText, Doc2Vec & K-Means Clustering

Uses the **same corpus, the same train / dev / test splits, and the same
`<UNK>`-folded vocabulary** as Assignment 4 (which in turn matches Assignment 3's
corpus: IndicCorpV2, one language, length-stratified dev/test of 1000
sentences each, training pool of 1,000,000 sentences). Re-running the
identical collection + split code with the same `RANDOM_SEED` reproduces the
exact same splits, so everything here lines up token-for-token with
Assignment 4's n-gram vocabulary.

For that training data, this notebook:

1. Trains **Word2Vec**, **FastText**, and **Doc2Vec** (gensim, following the
   tutorials linked in the assignment PDF).
2. **K-means clusters** the training vocabulary's word vectors and reports the
   20 words closest to each cluster centroid.
3. Represents every dev/test sentence as a vector — via averaged word
   embeddings (Word2Vec and FastText) and via Doc2Vec — and, for each one,
   finds its **closest sentence** in the training set under each
   representation.


## 0. Configuration

In [1]:
LANGUAGE = "hin_Deva"

DEV_SIZE = 1000
TEST_SIZE = 1000

TRAIN_SIZE = 1_000_000

UNK_MIN_FREQ = 2

RANDOM_SEED = 42

# --- Embedding hyperparameters (Word2Vec / FastText / Doc2Vec) ---
VECTOR_SIZE = 100
WINDOW_SIZE = 5
EPOCHS = 5
DOC2VEC_EPOCHS = 10
WORKERS = 4          # bump this up to os.cpu_count() if your machine has more cores
SKIP_GRAM = 1        # 1 = skip-gram, 0 = CBOW

GENSIM_MIN_COUNT = 1

# --- K-means clustering ---
NUM_CLUSTERS = 50
WORDS_PER_CLUSTER = 20   # "20 words closest to each cluster centroid", per the assignment

# Set this to True first to sanity-check the whole notebook quickly on tiny
# data (well under a minute), then set it back to False (default) to run the
# real assignment numbers above before you submit.
QUICK_TEST = False

if QUICK_TEST:
    TRAIN_SIZE = 8_000
    DEV_SIZE = 200
    TEST_SIZE = 200
    UNK_MIN_FREQ = 2
    NUM_CLUSTERS = 10
    EPOCHS = 3
    DOC2VEC_EPOCHS = 5

print("Language:", LANGUAGE)
print("QUICK_TEST:", QUICK_TEST)
print("Train size:", TRAIN_SIZE)
print("Dev/Test size:", DEV_SIZE, "/", TEST_SIZE)
print("Vector size:", VECTOR_SIZE, "| Num clusters:", NUM_CLUSTERS)


Language: hin_Deva
QUICK_TEST: False
Train size: 1000000
Dev/Test size: 1000 / 1000
Vector size: 100 | Num clusters: 50


In [2]:
!pip install datasets tqdm gensim scikit-learn pandas numpy

import os
import re
import math
import random
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from datasets import load_dataset
from tqdm.notebook import tqdm

from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from sklearn.cluster import KMeans

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## 1. Rebuild the corpus & length-stratified splits (identical to Assignment 4)

Same sentence-collection and length-stratified train/dev/test split code used
in Assignments 3 & 4, with the same `RANDOM_SEED`. Running it again with the
same `LANGUAGE`, `DEV_SIZE`, and `TEST_SIZE` reproduces the same splits, so
train/dev/test here are the exact same sentences Assignment 4 used.

In [3]:
SENT_SPLIT_PATTERN = re.compile(r'[।!?.\n]+')

def get_sentences(text):
    sentences = SENT_SPLIT_PATTERN.split(text)
    return [s.strip() for s in sentences if len(s.strip()) > 5]


def collect_sentences(language, target_count):
    """Stream paragraphs from IndicCorpV2 and collect `target_count` sentences."""
    dataset = load_dataset("ai4bharat/IndicCorpV2", split=language, streaming=True)
    sentences = []
    pbar = tqdm(total=target_count, desc=f"Collecting sentences ({language})")
    for sample in dataset:
        for sent in get_sentences(sample["text"]):
            sentences.append(sent)
            pbar.update(1)
            if len(sentences) >= target_count:
                pbar.close()
                return sentences
    pbar.close()
    print(f"WARNING: only found {len(sentences)} sentences, wanted {target_count}")
    return sentences


In [4]:
def stratified_length_split(sentences, dev_size, test_size, n_bins=20, seed=42):
    rng = random.Random(seed)
    lengths = np.array([len(s.split()) for s in sentences])

    # Bin edges from the quantiles of the length distribution -> bins have
    # roughly equal sentence counts, spanning short to long sentences.
    quantiles = np.linspace(0, 1, n_bins + 1)
    edges = np.unique(np.quantile(lengths, quantiles))
    bin_ids = np.digitize(lengths, edges[1:-1], right=True)

    indices_by_bin = defaultdict(list)
    for idx, b in enumerate(bin_ids):
        indices_by_bin[b].append(idx)
    for b in indices_by_bin:
        rng.shuffle(indices_by_bin[b])

    total = len(sentences)
    dev_idx, test_idx = [], []

    for b, idxs in indices_by_bin.items():
        share = len(idxs) / total
        n_dev = round(share * dev_size)
        n_test = round(share * test_size)
        dev_idx.extend(idxs[:n_dev])
        test_idx.extend(idxs[n_dev:n_dev + n_test])

    dev_idx = set(dev_idx)
    test_idx = set(test_idx)
    # Guard against rounding drift so we land on exactly dev_size / test_size.
    remaining = [i for i in range(total) if i not in dev_idx and i not in test_idx]
    rng.shuffle(remaining)
    while len(dev_idx) < dev_size:
        dev_idx.add(remaining.pop())
    while len(test_idx) < test_size:
        test_idx.add(remaining.pop())

    dev_sentences = [sentences[i] for i in dev_idx]
    test_sentences = [sentences[i] for i in test_idx]
    train_pool = [sentences[i] for i in range(total) if i not in dev_idx and i not in test_idx]
    rng.shuffle(train_pool)
    return dev_sentences, test_sentences, train_pool


In [5]:
total_needed = TRAIN_SIZE + DEV_SIZE + TEST_SIZE
all_sentences = collect_sentences(LANGUAGE, total_needed)

dev_sentences, test_sentences, train_pool = stratified_length_split(
    all_sentences, DEV_SIZE, TEST_SIZE, seed=RANDOM_SEED
)

train_sentences = train_pool[:TRAIN_SIZE]
if len(train_pool) < TRAIN_SIZE:
    print(f"WARNING: requested {TRAIN_SIZE} training sentences but pool only has {len(train_pool)}")

print(f"Train: {len(train_sentences)} | Dev: {len(dev_sentences)} | Test: {len(test_sentences)}")


Train: 999998 | Dev: 1001 | Test: 1001


## 2. Tokenization & vocabulary (identical to Assignment 4)

Same whitespace tokenization and `<UNK>`-folding as Assignment 4: training
words occurring fewer than `UNK_MIN_FREQ` times become a single `<UNK>`
token, and the same replacement is applied to dev/test. This keeps
Word2Vec/FastText/Doc2Vec's vocabulary identical to the one Assignment 4's
n-gram models were built on (FastText can still synthesize a reasonable
vector for anything folded into `<UNK>`, or for genuinely unseen words later,
via its character n-grams).

In [6]:
def tokenize(sentence):
    return sentence.split()


def build_vocab(token_lists, min_freq):
    freq = Counter(w for toks in token_lists for w in toks)
    vocab = {w for w, c in freq.items() if c >= min_freq}
    vocab.add("<UNK>")
    vocab.add("</s>")
    return vocab, freq


def apply_unk(token_lists, vocab):
    return [[w if w in vocab else "<UNK>" for w in toks] for toks in token_lists]


In [7]:
train_tok_raw = [tokenize(s) for s in tqdm(train_sentences, desc="Tokenizing train")]
dev_tok_raw = [tokenize(s) for s in dev_sentences]
test_tok_raw = [tokenize(s) for s in test_sentences]

vocab, train_word_freq = build_vocab(train_tok_raw, UNK_MIN_FREQ)

train_tok = apply_unk(train_tok_raw, vocab)
dev_tok = apply_unk(dev_tok_raw, vocab)
test_tok = apply_unk(test_tok_raw, vocab)

V = len(vocab)
print(f"Vocabulary size V (incl. <UNK>, </s>; excl. <s>): {V}")
print(f"Total training tokens (pre-UNK): {sum(len(t) for t in train_tok_raw)}")


Tokenizing train:   0%|          | 0/999998 [00:00<?, ?it/s]

Vocabulary size V (incl. <UNK>, </s>; excl. <s>): 162717
Total training tokens (pre-UNK): 17201602


## 3a. Train Word2Vec

Following gensim's
[Word2Vec tutorial](https://radimrehurek.com/gensim/auto_examples/tutorials/run_word2vec.html),
trained on `train_tok` (the same `<UNK>`-folded tokenization as Assignment 4).

In [8]:
w2v_model = Word2Vec(
    sentences=train_tok,
    vector_size=VECTOR_SIZE,
    window=WINDOW_SIZE,
    min_count=GENSIM_MIN_COUNT,
    workers=WORKERS,
    sg=SKIP_GRAM,
    epochs=EPOCHS,
    seed=RANDOM_SEED,
)
print(f"Word2Vec vocabulary size: {len(w2v_model.wv.index_to_key)}")

example_word = w2v_model.wv.index_to_key[0]
print(f"Most similar words to {example_word!r}:")
print(w2v_model.wv.most_similar(example_word, topn=5))


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Word2Vec vocabulary size: 162716
Most similar words to 'के':
[('केे', 0.7687097787857056), ('इसके', 0.7188058495521545), ('और', 0.691917359828949), ('में', 0.6883066296577454), ('इनके', 0.6849736571311951)]


## 3b. Train FastText

Following gensim's
[FastText tutorial](https://radimrehurek.com/gensim/auto_examples/tutorials/run_fasttext.html).
FastText also learns character n-gram vectors, so — unlike Word2Vec — it can
still produce a vector for a word it never saw during training.

In [9]:
ft_model = FastText(
    sentences=train_tok,
    vector_size=VECTOR_SIZE,
    window=WINDOW_SIZE,
    min_count=GENSIM_MIN_COUNT,
    workers=WORKERS,
    sg=SKIP_GRAM,
    epochs=EPOCHS,
    seed=RANDOM_SEED,
)
print(f"FastText vocabulary size: {len(ft_model.wv.index_to_key)}")

example_word = ft_model.wv.index_to_key[0]
print(f"Most similar words to {example_word!r}:")
print(ft_model.wv.most_similar(example_word, topn=5))


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

FastText vocabulary size: 162716
Most similar words to 'के':
[('केे', 0.7909438610076904), ('केेे', 0.7787131071090698), ('\u200bइसके', 0.7538429498672485), ('(उसके', 0.7467828989028931), ('गए,इसके', 0.7454159259796143)]


## 3c. Train Doc2Vec

Following gensim's
[Doc2Vec tutorial](https://radimrehurek.com/gensim/auto_examples/tutorials/run_doc2vec_lee.html).
Every training sentence becomes a `TaggedDocument` tagged with its own index,
so after training `doc2vec_model.dv[i]` is that sentence's learned vector —
no need to re-`infer_vector` it later.

In [10]:
train_tagged_docs = [
    TaggedDocument(words=tokens, tags=[i]) for i, tokens in enumerate(train_tok)
]

doc2vec_model = Doc2Vec(
    vector_size=VECTOR_SIZE,
    window=WINDOW_SIZE,
    min_count=GENSIM_MIN_COUNT,
    workers=WORKERS,
    epochs=DOC2VEC_EPOCHS,
    seed=RANDOM_SEED,
)
doc2vec_model.build_vocab(train_tagged_docs)
doc2vec_model.train(
    train_tagged_docs,
    total_examples=doc2vec_model.corpus_count,
    epochs=doc2vec_model.epochs,
)
print(f"Doc2Vec vocabulary size: {len(doc2vec_model.wv.index_to_key)}")
print(f"Doc2Vec learned {len(doc2vec_model.dv)} document vectors")


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Doc2Vec vocabulary size: 162716
Doc2Vec learned 999998 document vectors


## 4. K-means clustering of the training vocabulary

We cluster **Word2Vec's** word vectors (its `index_to_key` vocabulary) into
`NUM_CLUSTERS` clusters, then for each cluster report the
`WORDS_PER_CLUSTER` words whose vectors are closest (Euclidean distance,
matching K-means' own objective) to that cluster's centroid.

In [11]:
vocab_words_w2v = w2v_model.wv.index_to_key
word_vectors = w2v_model.wv.vectors  # same order as vocab_words_w2v

kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
cluster_assignments = kmeans.fit_predict(word_vectors)
centroids = kmeans.cluster_centers_

cluster_top_words = {}
for c in range(NUM_CLUSTERS):
    dists = np.linalg.norm(word_vectors - centroids[c], axis=1)
    closest_idx = np.argsort(dists)[:WORDS_PER_CLUSTER]
    cluster_top_words[c] = [(vocab_words_w2v[i], float(dists[i])) for i in closest_idx]

cluster_sizes = pd.Series(cluster_assignments).value_counts().sort_index()

for c in range(NUM_CLUSTERS):
    words_str = ", ".join(w for w, _ in cluster_top_words[c])
    print(f"Cluster {c:>3} (size={cluster_sizes.get(c, 0):>5}): {words_str}")


Cluster   0 (size=19477): लोगों’, (शैतानी, पीसेज, क़ियामत, pigeon, ऑक्सफ़ोर्ड, Normal, गाँववाले, e), स्यापा, (PC), दफ़्न, आइं, कवयित्री,, स्पाइडर', नग्नता, बंदरिया, न्यूडिटी, Derive, फ़िरोज़ा
Cluster   1 (size=  926): रेवाड़ी, शमसाबाद, मोहम्मदाबाद, बिसौली, कलानौर, मोहम्मदी, बड़कागांव, कांधला, खटीमा, गोपीगंज, सादाबाद, बिधनू, कीर्तिनगर, बड़वानी, बड़हिया, जयसिंहपुर, बावल, बांसी, सहावर, जलेसर
Cluster   2 (size=10579): म्याजलार,, हाटपिपलिया, गग्गी, इशमीत, मकरानी,, बगवां),, राके, भारतसिंह, साजन,, मेहसी,, नगौला, बैजू,, चन्दपाल, शिवचरन, नवले,, गुरलाभ, डनसेना,, मोहतरा, बरिदरमीत, इंद्रदत्त
Cluster   3 (size= 1218): पुर्नवास, कम्प्यूटरीकरण, ई-मित्र, प्रशासनों, त्रिढयान्वयन, अंकेक्षण, कियान्वयन, पुस्तकालयों, युक्तियुक्तकरण, आवासन, कार्यों,, संनिर्माण, सुदृढ़ीकरण, डीएमएफटी, सहायतार्थ, गोठान, निबंधित, पभावी, दीदियों, पुनर्निधारण
Cluster   4 (size= 3752): खडगे, ''प्रधानमंत्री, “ममता, ‘सरकारी, बोल्शेविकों, कालसे, शेट्टर, ट्वीटों, ओ'ब्रायन, मोहिउद्दीन, ''केंद्र, एसोशिएशन, वक्त-वक्त, मायवती, थिनले, पत्र', र

## 5. Sentence representations

Three ways to turn a sentence into a single vector:

1. **Word2Vec-average** — mean of the Word2Vec vectors of its words (all of
   them are in-vocabulary here, since `<UNK>` folding already happened, and
   `<UNK>` itself gets a trained vector like any other token).
2. **FastText-average** — mean of the FastText vectors of its words.
3. **Doc2Vec** — the model's own learned document vector for training
   sentences (`doc2vec_model.dv[i]`), and `infer_vector` for dev/test
   sentences (which the model never saw during training).

In [12]:
def w2v_average_vector(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return np.zeros(model.vector_size, dtype=np.float32)
    return np.mean(vecs, axis=0)


def ft_average_vector(tokens, model):
    vecs = []
    for w in tokens:
        try:
            vecs.append(model.wv[w])
        except KeyError:
            continue  # extremely rare: e.g. an empty/degenerate token
    if not vecs:
        return np.zeros(model.vector_size, dtype=np.float32)
    return np.mean(vecs, axis=0)


def build_vectors(token_lists, vector_fn, model, desc):
    return np.array(
        [vector_fn(tokens, model) for tokens in tqdm(token_lists, desc=desc)],
        dtype=np.float32,
    )


print("Building Word2Vec-average sentence vectors...")
train_vecs_w2v = build_vectors(train_tok, w2v_average_vector, w2v_model, "train (w2v-avg)")
dev_vecs_w2v = build_vectors(dev_tok, w2v_average_vector, w2v_model, "dev (w2v-avg)")
test_vecs_w2v = build_vectors(test_tok, w2v_average_vector, w2v_model, "test (w2v-avg)")

print("Building FastText-average sentence vectors...")
train_vecs_ft = build_vectors(train_tok, ft_average_vector, ft_model, "train (ft-avg)")
dev_vecs_ft = build_vectors(dev_tok, ft_average_vector, ft_model, "dev (ft-avg)")
test_vecs_ft = build_vectors(test_tok, ft_average_vector, ft_model, "test (ft-avg)")

print("Collecting Doc2Vec training vectors (already learned during training)...")
train_vecs_d2v = np.array([doc2vec_model.dv[i] for i in range(len(train_tok))], dtype=np.float32)

print("Inferring Doc2Vec vectors for dev/test (unseen during training)...")
dev_vecs_d2v = np.array(
    [doc2vec_model.infer_vector(tokens, epochs=DOC2VEC_EPOCHS) for tokens in tqdm(dev_tok, desc="dev (doc2vec infer)")],
    dtype=np.float32,
)
test_vecs_d2v = np.array(
    [doc2vec_model.infer_vector(tokens, epochs=DOC2VEC_EPOCHS) for tokens in tqdm(test_tok, desc="test (doc2vec infer)")],
    dtype=np.float32,
)

print("Shapes:", train_vecs_w2v.shape, train_vecs_ft.shape, train_vecs_d2v.shape)


Building Word2Vec-average sentence vectors...


train (w2v-avg):   0%|          | 0/999998 [00:00<?, ?it/s]

dev (w2v-avg):   0%|          | 0/1001 [00:00<?, ?it/s]

test (w2v-avg):   0%|          | 0/1001 [00:00<?, ?it/s]

Building FastText-average sentence vectors...


...

test (ft-avg):   0%|          | 0/1001 [00:00<?, ?it/s]

Inferring Doc2Vec vectors for dev/test (unseen during training)...


dev (doc2vec infer):   0%|          | 0/1001 [00:00<?, ?it/s]

test (doc2vec infer):   0%|          | 0/1001 [00:00<?, ?it/s]

Shapes: (999998, 100) (999998, 100) (999998, 100)


## 6. Finding the closest training sentence

For every dev/test sentence, under each representation, we find the training
sentence whose vector has the highest **cosine similarity**. This is done
with a single normalized-vector matrix multiplication per query batch
(`query_batch @ train_matrix.T`) rather than a Python loop over 1,000,000
training sentences, so it stays fast even at this corpus scale.

In [13]:
def normalize_rows(mat, eps=1e-10):
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms < eps] = eps
    return mat / norms


def find_nearest(query_vecs, reference_vecs_normalized, batch_size=500):
    """Returns (best_idx, best_sim) arrays, one entry per query vector."""
    query_norm = normalize_rows(query_vecs)
    best_idx = np.zeros(len(query_vecs), dtype=np.int64)
    best_sim = np.zeros(len(query_vecs), dtype=np.float32)
    for start in range(0, len(query_norm), batch_size):
        end = start + batch_size
        sims = query_norm[start:end] @ reference_vecs_normalized.T  # (batch, n_train)
        idx = np.argmax(sims, axis=1)
        best_idx[start:end] = idx
        best_sim[start:end] = sims[np.arange(len(idx)), idx]
    return best_idx, best_sim


train_vecs_w2v_n = normalize_rows(train_vecs_w2v)
train_vecs_ft_n = normalize_rows(train_vecs_ft)
train_vecs_d2v_n = normalize_rows(train_vecs_d2v)

METHODS = {
    "Word2Vec-avg": (dev_vecs_w2v, test_vecs_w2v, train_vecs_w2v_n),
    "FastText-avg": (dev_vecs_ft, test_vecs_ft, train_vecs_ft_n),
    "Doc2Vec": (dev_vecs_d2v, test_vecs_d2v, train_vecs_d2v_n),
}

nearest_results = {}  # (split, method) -> (idx array, sim array)
for method, (dev_q, test_q, train_ref) in METHODS.items():
    nearest_results[("dev", method)] = find_nearest(dev_q, train_ref)
    nearest_results[("test", method)] = find_nearest(test_q, train_ref)
    print(f"Done: {method}")


Done: Word2Vec-avg
Done: FastText-avg
Done: Doc2Vec


## 7. Results

A table of every dev/test sentence, its nearest training sentence under each
representation, and the cosine similarity — plus a few printed examples so
you can eyeball the quality of the matches.

In [14]:
rows = []
for split, sentences in [("dev", dev_sentences), ("test", test_sentences)]:
    for i, sent in enumerate(sentences):
        row = {"split": split, "sentence": sent}
        for method in METHODS:
            idx_arr, sim_arr = nearest_results[(split, method)]
            row[f"{method} - nearest"] = train_sentences[idx_arr[i]]
            row[f"{method} - similarity"] = float(sim_arr[i])
        rows.append(row)

nearest_df = pd.DataFrame(rows)
nearest_df.to_csv("sentence_nearest_neighbors.csv", index=False)
print(f"Saved full results for {len(nearest_df)} sentences to sentence_nearest_neighbors.csv")
nearest_df.head(10)


Saved full results for 2002 sentences to sentence_nearest_neighbors.csv


,split,sentence,Word2Vec-avg - nearest,Word2Vec-avg - similarity,FastText-avg - nearest,FastText-avg - similarity,Doc2Vec - nearest,Doc2Vec - similarity
0,dev,सच इसलिए भी क्योंकि जरूरतमंदों के लिए मददगारों...,विचाराधीन बंदियों के लिए यह समस्या और भी ज़्याद...,0.964558,इसलिए ऐसे लोगों को परेशान होने की जरूरत नहीं है,0.957937,मगर मोदी के लिए नाकामयाबी की तो गुंजाइश ही नही...,0.546202
1,dev,कर्क राशि:- कर्क राशि के जातक आज पेपर वर्क पर ...,"Horoscope Today, 02 February 2021: मिथुन राशि ...",0.948877,मिथुन राशि – मिथुन राशि के जातक बिजनेस में पार...,0.956010,GameSpot,0.729522
2,dev,हरियाणा में रात 10:00 बजे से सुबह 5:00 बजे के ...,अरुणाचल प्रदेश में कोविड-19 के प्रसार को रोकने...,0.967539,अरुणाचल प्रदेश में कोविड-19 के प्रसार को रोकने...,0.967337,इस बीच सभी गैर जरूरी गतिविधियों के लिए व्यक्ति...,0.679553
3,dev,रूस के राष्ट्रपति व्लादीमिर पुतिन ने यूक्रेन क...,नाटो ने भी कोसोवो के इस फ़ैसले पर चिंता जताई है,0.959765,यहाँ रूसी राष्ट्रपति व्लादिमीर पुतिन ने भी अम...,0.961091,-पुलिस ने शव को पोस्टमार्टम के लिए भेज दिया है,0.894723
4,dev,एएसआई शहीद कुलदीप सिंह का शनिवार को उनके पैतृक...,पीपलकोटी के स्यूंण गांव के शहीद सुरजीत सिंह रा...,0.963757,पीपलकोटी के स्यूंण गांव के शहीद सुरजीत सिंह रा...,0.971605,फिल्म का निर्देशन मेघना गुलजार ने किया है,0.785822
5,dev,सिंधुताई पास 36 बहुएं हैं और 272 दामाद हैं,हमारे पास 57 विधायक हैं,0.922492,हमारे पास 170 से ज्यादा विधायक हैं,0.928429,वहीं बीजेपी के पास 72 विधायक हैं,0.686670
6,dev,सोमवार का दिन पारिवारिक जिम्मेदारियों को निभात...,इस राशि में आने वाले जातक रोमांस और सच्चे प्या...,0.920066,बुधवार के दिन बेटी को विदा करने से आपकी बेटी क...,0.928415,शनिवार का दिन मानसिक अशांति तथा खर्च की अधिकता...,0.560205
7,dev,मूंग दाल के पानी में होते है ये गुण,चिकन पॉक्स होने पर नीम की पत्तियां पानी में उब...,0.946294,यह आलू टैंगी मिर्च और नींबू के साथ पकाए जाते हैं,0.939300,rgfindia,0.755244
8,dev,"भारत में तकनीकी शिक्षा कई भागों में-डिप्लोमा, ...",क्रिसलिस (पूववर्ती ईजेड विद्या) स्कूली शिक्षा ...,0.950089,आज के इकोनॉमिक लॉ या आर्थिक विधियों के समतुल्य...,0.959112,डिग्री का इंतजार है,0.689362
9,dev,आरोप पत्र में कुल 24 आरोपियों के नाम हैं,भंवरी मामले में आरोप पत्र सात तक,0.938955,भंवरी मामले में आरोप पत्र सात तक,0.938959,वह पोस्टमार्टम हाउस प्रभारी भी थे,0.601989


In [15]:
# Eyeball a handful of dev examples across all three methods
for i in range(min(5, len(dev_sentences))):
    print(f"\nDev sentence: {dev_sentences[i]}")
    for method in METHODS:
        idx_arr, sim_arr = nearest_results[("dev", method)]
        print(f"  [{method:12s} sim={sim_arr[i]:.3f}] {train_sentences[idx_arr[i]]}")



Dev sentence: सच इसलिए भी क्योंकि जरूरतमंदों के लिए मददगारों की संख्या भी कम नहीं
  [Word2Vec-avg sim=0.965] विचाराधीन बंदियों के लिए यह समस्या और भी ज़्यादा है क्योंकि उन पर किसी काम को करने का कोई दबाव भी नहीं होता और जेल में उनकी अवधि भी निश्चित नहीं होती
  [FastText-avg sim=0.958] इसलिए ऐसे लोगों को परेशान होने की जरूरत नहीं है
  [Doc2Vec      sim=0.546] मगर मोदी के लिए नाकामयाबी की तो गुंजाइश ही नहीं है

Dev sentence: कर्क राशि:- कर्क राशि के जातक आज पेपर वर्क पर ज्यादा ध्यान देंगे
  [Word2Vec-avg sim=0.949] Horoscope Today, 02 February 2021: मिथुन राशि के जातक समझदारी से निवेश करें, मेष राशि वाले शराब से रहें दूर Horoscope Today (आज का राशिफल) 02 February: मीन राशि के जातक दूसरों के लिए ख़राब नीयत रखना मानसिक तनाव को जन्म दे सकता है
  [FastText-avg sim=0.956] मिथुन राशि – मिथुन राशि के जातक बिजनेस में पार्टनरशिप से बचेंगे तो  मिलेगा लाभ जल्दबाजी करने से बचें
  [Doc2Vec      sim=0.730] GameSpot

Dev sentence: हरियाणा में रात 10:00 बजे से सुबह 5:00 बजे के बीच सभी गैर-जरूरी गतिविधियो

## 8. Do the methods agree with each other?

How often do Word2Vec-average, FastText-average, and Doc2Vec point at the
*same* nearest training sentence? High agreement suggests the sentence-level
semantics they capture are similar; low agreement suggests they're picking up
on different things (e.g. lexical overlap vs. more holistic sentence
structure).

In [16]:
def pairwise_agreement(split):
    methods = list(METHODS.keys())
    n = len(nearest_results[(split, methods[0])][0])
    print(f"\n=== {split} set (n={n}) ===")
    for i in range(len(methods)):
        for j in range(i + 1, len(methods)):
            idx_i, _ = nearest_results[(split, methods[i])]
            idx_j, _ = nearest_results[(split, methods[j])]
            agree = np.mean(idx_i == idx_j) * 100
            print(f"  {methods[i]:12s} vs {methods[j]:12s}: {agree:5.1f}% same nearest sentence")

pairwise_agreement("dev")
pairwise_agreement("test")



=== dev set (n=1001) ===
  Word2Vec-avg vs FastText-avg:  50.0% same nearest sentence
  Word2Vec-avg vs Doc2Vec     :   0.8% same nearest sentence
  FastText-avg vs Doc2Vec     :   0.7% same nearest sentence

=== test set (n=1001) ===
  Word2Vec-avg vs FastText-avg:  50.9% same nearest sentence
  Word2Vec-avg vs Doc2Vec     :   1.1% same nearest sentence
  FastText-avg vs Doc2Vec     :   1.3% same nearest sentence
